In [1]:
import pandas as pd
import numpy as np
import seaborn as sns

In [23]:
secilecek_sutunlar = ['TARIH', 'KURTARMA_TURU', 'CIKIS_SAATI', 'VARIS_SURESI', 'MAHALLE', 'ILCE']
df = pd.read_excel("2024-yili-arama-kurtarma.xlsx", usecols=secilecek_sutunlar)

In [24]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 12828 entries, 0 to 12827
Data columns (total 6 columns):
 #   Column         Non-Null Count  Dtype 
---  ------         --------------  ----- 
 0   TARIH          12828 non-null  object
 1   KURTARMA_TURU  12828 non-null  object
 2   CIKIS_SAATI    12828 non-null  object
 3   VARIS_SURESI   12760 non-null  object
 4   MAHALLE        12803 non-null  object
 5   ILCE           12828 non-null  object
dtypes: object(6)
memory usage: 601.4+ KB


In [25]:
df.head()

,TARIH,KURTARMA_TURU,CIKIS_SAATI,VARIS_SURESI,MAHALLE,ILCE
0,1.01.2024,112 ACİL SAĞLIK/AMBULANS,00:41,00:04,GÜNEY,KONAK
1,1.01.2024,112 ACİL SAĞLIK/AMBULANS,00:50,NaN,GÜMÜŞPALA,BAYRAKLI
2,1.01.2024,TRAFİK KAZASI,01:26,00:04,KURUÇEŞME,BUCA
3,1.01.2024,112 ACİL SAĞLIK/AMBULANS,01:58,00:07,BOSTANLI,KARŞIYAKA
4,1.01.2024,YÜZÜK KESME,02:42,00:03,KAZIMDİRİK,BORNOVA


In [26]:
df.columns = ['tarih','kurtarma_tipi', 'cikis_saati', 'varis_suresi', 'mahalle', 'ilce']
df

,tarih,kurtarma_tipi,cikis_saati,varis_suresi,mahalle,ilce
0,1.01.2024,112 ACİL SAĞLIK/AMBULANS,00:41,00:04,GÜNEY,KONAK
1,1.01.2024,112 ACİL SAĞLIK/AMBULANS,00:50,NaN,GÜMÜŞPALA,BAYRAKLI
2,1.01.2024,TRAFİK KAZASI,01:26,00:04,KURUÇEŞME,BUCA
3,1.01.2024,112 ACİL SAĞLIK/AMBULANS,01:58,00:07,BOSTANLI,KARŞIYAKA
4,1.01.2024,YÜZÜK KESME,02:42,00:03,KAZIMDİRİK,BORNOVA
...,...,...,...,...,...,...
12823,2024-12-31 00:00:00,HAYVAN (KEDİ) KURTARMA,17:14,00:04,YAMANLAR,BAYRAKLI
12824,2024-12-31 00:00:00,HAYVAN (KEDİ) KURTARMA,18:54,00:03,EGEMENLİK,BORNOVA
12825,2024-12-31 00:00:00,ARAÇTA MAHSUR KALMA,19:03,00:04,AOSB,ÇİĞLİ
12826,2024-12-31 00:00:00,KAPI AÇMA (ACİL DURUM),20:06,00:04,HACI BEŞİR,BAYINDIR


In [27]:
def sureyi_hesapla(x):
    # Eğer veri string değilse veya boşsa (NaN) doğrudan 0 veya None döndür
    if not isinstance(x, str) or ":" not in str(x):
        return 0 
    
    parcalar = x.split(':')
    # HH:MM formatını dakikaya çeviriyoruz
    return int(parcalar[0]) * 60 + int(parcalar[1])

# Fonksiyonu güvenli bir şekilde uygula
df['toplam_dakika'] = df['varis_suresi'].apply(sureyi_hesapla)

# Kontrol için ilk 5 satırı yazdır
print(df[['varis_suresi', 'toplam_dakika']].head())

  varis_suresi  toplam_dakika
0        00:04              4
1          NaN              0
2        00:04              4
3        00:07              7
4        00:03              3


In [66]:
df.head()
df.describe()

,toplam_dakika
count,12759.000000
mean,6.854064
std,18.906326
min,1.000000
25%,3.000000
50%,5.000000
75%,6.000000
max,981.000000


In [30]:
df = df[df['toplam_dakika'] > 0]

In [31]:
df.head()

,tarih,kurtarma_tipi,cikis_saati,varis_suresi,mahalle,ilce,toplam_dakika
0,1.01.2024,112 ACİL SAĞLIK/AMBULANS,00:41,00:04,GÜNEY,KONAK,4
2,1.01.2024,TRAFİK KAZASI,01:26,00:04,KURUÇEŞME,BUCA,4
3,1.01.2024,112 ACİL SAĞLIK/AMBULANS,01:58,00:07,BOSTANLI,KARŞIYAKA,7
4,1.01.2024,YÜZÜK KESME,02:42,00:03,KAZIMDİRİK,BORNOVA,3
5,1.01.2024,HAYVAN (KÖPEK) KURTARMA,09:14,00:05,HÜRRİYET,GAZİEMİR,5


In [33]:
df['toplam_dakika'].mean() #izmir geneli ortalama mudahale suresi

np.float64(6.8540637981033)

In [36]:
ilce_analizi = df.groupby('ilce')['toplam_dakika'].mean().sort_values()

In [37]:
ilce_analizi

ilce
KINIK            4.162162
ALİAĞA           5.036364
FOÇA             5.127119
ÇEŞME            5.530000
BAYRAKLI         5.574794
BORNOVA          5.729522
NARLIDERE        5.785714
MENEMEN          5.831754
KARŞIYAKA        5.854512
ÇİĞLİ            5.885135
TORBALI          5.902669
BERGAMA          5.922794
KONAK            6.012242
TİRE             6.062147
GAZİEMİR         6.087719
BALÇOVA          6.103093
SELÇUK           6.375000
SEFERİHİSAR      6.381643
KARABURUN        6.725000
GÜZELBAHÇE       6.729508
ÖDEMİŞ           7.000000
DİKİLİ           7.265896
BUCA             7.508516
KARABAĞLAR       7.608074
KEMALPAŞA        8.075758
MENDERES         8.242798
URLA             8.375000
BEYDAĞ           8.500000
BAYINDIR         8.969697
KİRAZ            9.210526
İL DIŞI        249.043478
Name: toplam_dakika, dtype: float64

In [43]:
print(df.groupby('ilce')['toplam_dakika'].mean().sort_values(ascending=False))

ilce
İL DIŞI        249.043478
KİRAZ            9.210526
BAYINDIR         8.969697
BEYDAĞ           8.500000
URLA             8.375000
MENDERES         8.242798
KEMALPAŞA        8.075758
KARABAĞLAR       7.608074
BUCA             7.508516
DİKİLİ           7.265896
ÖDEMİŞ           7.000000
GÜZELBAHÇE       6.729508
KARABURUN        6.725000
SEFERİHİSAR      6.381643
SELÇUK           6.375000
BALÇOVA          6.103093
GAZİEMİR         6.087719
TİRE             6.062147
KONAK            6.012242
BERGAMA          5.922794
TORBALI          5.902669
ÇİĞLİ            5.885135
KARŞIYAKA        5.854512
MENEMEN          5.831754
NARLIDERE        5.785714
BORNOVA          5.729522
BAYRAKLI         5.574794
ÇEŞME            5.530000
FOÇA             5.127119
ALİAĞA           5.036364
KINIK            4.162162
Name: toplam_dakika, dtype: float64


In [41]:
df['toplam_dakika'].describe()

count    12759.000000
mean         6.854064
std         18.906326
min          1.000000
25%          3.000000
50%          5.000000
75%          6.000000
max        981.000000
Name: toplam_dakika, dtype: float64

In [44]:
# Kiraz ilçesine ait verileri ayıklayalım
df_kiraz = df[df['ilce'] == 'KİRAZ']

# Kiraz'da en çok hangi vakalar yaşanmış?
print("Kiraz Vaka Dağılımı:")
print(df_kiraz['kurtarma_tipi'].value_counts())

Kiraz Vaka Dağılımı:
kurtarma_tipi
TRAFİK KAZASI               10
HAYVAN (KEDİ) KURTARMA       8
HAYVAN (DİĞER) KURTARMA      4
ASANSÖRDEN KURTARMA          3
AKS AMBULANS HASTA SEVKİ     3
BİNADA MAHSUR KALMA          2
HAYVAN (KÖPEK) KURTARMA      2
AĞAÇ DEVRİLMESİ              2
İNTİHAR GİRİŞİMİ             2
DİĞER                        1
KAPI AÇMA (ACİL DURUM)       1
Name: count, dtype: int64


In [45]:
# Buca ilçesine ait verileri ayıklayalım
df_buca = df[df['ilce'] == 'BUCA']

# Kiraz'da en çok hangi vakalar yaşanmış?
print("Buca Vaka Dağılımı:")
print(df_buca['kurtarma_tipi'].value_counts())

Buca Vaka Dağılımı:
kurtarma_tipi
AKS AMBULANS HASTA SEVKİ                                392
HAYVAN (KEDİ) KURTARMA                                  365
KAPI AÇMA (ACİL DURUM)                                  108
TRAFİK KAZASI                                            73
HAYVAN (KUŞ VB.) KURTARMA                                53
ASANSÖRDEN KURTARMA                                      53
İNTİHAR GİRİŞİMİ                                         39
HAYVAN (DİĞER) KURTARMA                                  30
HAYVAN (KÖPEK) KURTARMA                                  21
BİNADA MAHSUR KALMA                                      16
AĞAÇ DEVRİLMESİ                                          15
DİĞER                                                    14
112 ACİL SAĞLIK/AMBULANS                                 11
YÜZÜK KESME                                              11
ÇATI/TABELA/PANO VB. UÇMASI                               9
ARAÇTA MAHSUR KALMA                                       8
DÜŞME 

In [46]:
def ilce_veri_ayikla(ilce_adi):
    df_ilce = df[df['ilce'] == ilce_adi]

    print(f"{ilce_adi} vaka dagilimi:")
    print(df_ilce["kurtarma_tipi"].value_counts())

In [47]:
ilce_veri_ayikla('KONAK')

KONAK vaka dagilimi:
kurtarma_tipi
HAYVAN (KEDİ) KURTARMA                  471
AKS AMBULANS HASTA SEVKİ                382
KAPI AÇMA (ACİL DURUM)                  134
112 ACİL SAĞLIK/AMBULANS                104
HAYVAN (KUŞ VB.) KURTARMA                85
TRAFİK KAZASI                            64
İNTİHAR GİRİŞİMİ                         43
DİĞER                                    42
ASANSÖRDEN KURTARMA                      40
HAYVAN (DİĞER) KURTARMA                  37
YÜZÜK KESME                              27
AĞAÇ DEVRİLMESİ                          22
ÇATI/TABELA/PANO VB. UÇMASI              22
BİNADA MAHSUR KALMA                      20
HAYVAN (KÖPEK) KURTARMA                  15
SUDA ARAMA KURTARMA (DENİZ GÖLET VB)     13
DÜŞME (DERE/ÇUKUR VB)                     9
SIKIŞMA                                   7
CESET ÇIKARMA                             5
ARAÇTA MAHSUR KALMA                       4
SELDE/SU BASKININDA ARAMA KURTARMA        3
İŞ KAZASI                                

In [48]:
# Sadece trafik kazalarını filtreleyelim
df_trafik = df[df['kurtarma_tipi'] == 'TRAFİK KAZASI'].copy()

In [49]:
df_trafik

,tarih,kurtarma_tipi,cikis_saati,varis_suresi,mahalle,ilce,toplam_dakika
2,1.01.2024,TRAFİK KAZASI,01:26,00:04,KURUÇEŞME,BUCA,4
18,1.01.2024,TRAFİK KAZASI,21:25,00:07,EVKA-5,ÇİĞLİ,7
26,2.01.2024,TRAFİK KAZASI,09:09,00:05,GÜRPINAR,BORNOVA,5
30,2.01.2024,TRAFİK KAZASI,11:20,00:20,İĞDELİ,KİRAZ,20
37,2.01.2024,TRAFİK KAZASI,14:08,00:07,ATATÜRK,BUCA,7
...,...,...,...,...,...,...,...
12759,2024-12-29 00:00:00,TRAFİK KAZASI,19:14,00:05,KAZIMDİRİK,BORNOVA,5
12768,2024-12-30 00:00:00,TRAFİK KAZASI,00:23,00:02,ÇORAKLAR,ALİAĞA,2
12782,2024-12-30 00:00:00,TRAFİK KAZASI,12:05,00:04,ŞEHİTKEMAL,ALİAĞA,4
12808,2024-12-31 00:00:00,TRAFİK KAZASI,12:15,00:06,BİNBAŞI REŞATBEY,GAZİEMİR,6


In [50]:
print(f"Toplam Trafik Kazası Sayısı: {len(df_trafik)}")

Toplam Trafik Kazası Sayısı: 1123


In [51]:
# drop=True parametresi, eski karışık indekslerin yeni bir sütun olarak eklenmesini engeller.
df_trafik = df_trafik.reset_index(drop=True)

In [52]:
df_trafik

,tarih,kurtarma_tipi,cikis_saati,varis_suresi,mahalle,ilce,toplam_dakika
0,1.01.2024,TRAFİK KAZASI,01:26,00:04,KURUÇEŞME,BUCA,4
1,1.01.2024,TRAFİK KAZASI,21:25,00:07,EVKA-5,ÇİĞLİ,7
2,2.01.2024,TRAFİK KAZASI,09:09,00:05,GÜRPINAR,BORNOVA,5
3,2.01.2024,TRAFİK KAZASI,11:20,00:20,İĞDELİ,KİRAZ,20
4,2.01.2024,TRAFİK KAZASI,14:08,00:07,ATATÜRK,BUCA,7
...,...,...,...,...,...,...,...
1118,2024-12-29 00:00:00,TRAFİK KAZASI,19:14,00:05,KAZIMDİRİK,BORNOVA,5
1119,2024-12-30 00:00:00,TRAFİK KAZASI,00:23,00:02,ÇORAKLAR,ALİAĞA,2
1120,2024-12-30 00:00:00,TRAFİK KAZASI,12:05,00:04,ŞEHİTKEMAL,ALİAĞA,4
1121,2024-12-31 00:00:00,TRAFİK KAZASI,12:15,00:06,BİNBAŞI REŞATBEY,GAZİEMİR,6


In [53]:
print(df_trafik.groupby('ilce')['toplam_dakika'].mean().sort_values(ascending=False))

ilce
İL DIŞI        16.500000
KİRAZ          13.800000
BEYDAĞ         12.000000
ÖDEMİŞ         10.258065
TİRE            9.600000
GÜZELBAHÇE      9.375000
SELÇUK          9.290323
SEFERİHİSAR     8.250000
KEMALPAŞA       7.890244
URLA            7.827586
BAYINDIR        7.461538
DİKİLİ          7.272727
MENDERES        7.033898
TORBALI         6.689189
MENEMEN         6.666667
BERGAMA         6.600000
KINIK           6.111111
ALİAĞA          5.560976
BUCA            5.479452
ÇEŞME           5.344828
NARLIDERE       5.222222
BORNOVA         4.882353
ÇİĞLİ           4.863636
BAYRAKLI        4.630435
GAZİEMİR        4.575758
BALÇOVA         4.500000
KARABURUN       4.500000
KARŞIYAKA       4.488372
KARABAĞLAR      3.795455
FOÇA            3.600000
KONAK           3.500000
Name: toplam_dakika, dtype: float64


In [58]:
# '08:55' gibi bir veriden ilk iki karakteri (saati) alıp sayıya çeviriyoruz
df_trafik['saat'] = df_trafik['cikis_saati'].str.split(':').str[0].astype(int)

# Hangi saatte kaç kaza meydana gelmiş?
saatlik_kaza_sayisi = df_trafik['saat'].value_counts().sort_index()
print("Günün saatlerine göre kaza dağılımı:")
print(saatlik_kaza_sayisi)

Günün saatlerine göre kaza dağılımı:
saat
0     47
1     52
2     32
3     26
4     24
5     24
6     19
7     37
8     48
9     47
10    38
11    54
12    45
13    54
14    47
15    55
16    68
17    69
18    60
19    59
20    48
21    40
22    64
23    66
Name: count, dtype: int64


In [61]:
# Saatlere göre ortalama varış süresini hesaplayalım
saatlik_performans = df_trafik.groupby('saat')['toplam_dakika'].mean()

print("Saat bazlı ortalama müdahale süresi (dakika):")
print(saatlik_performans)

Saat bazlı ortalama müdahale süresi (dakika):
saat
0     5.723404
1     5.480769
2     5.250000
3     6.000000
4     5.125000
5     6.250000
6     6.052632
7     8.702703
8     7.437500
9     5.808511
10    6.184211
11    6.870370
12    6.844444
13    5.018519
14    7.468085
15    6.509091
16    5.750000
17    6.550725
18    7.000000
19    5.711864
20    5.916667
21    6.000000
22    5.765625
23    5.272727
Name: toplam_dakika, dtype: float64


In [63]:
# Sadece merkez ilçeleri ve hafta içini filtreleyerek bakalım
merkez_ilceler = ['KONAK', 'BORNOVA', 'KARŞIYAKA', 'BAYRAKLI', 'BUCA']
df_merkez = df_trafik[df_trafik['ilce'].isin(merkez_ilceler)].copy()

# Medyan üzerinden saatlik performans (Uç değerlerden etkilenmez)
print("Merkez İlçeler Saatlik Medyan Müdahale Süresi:")
print(df_merkez.groupby('saat')['toplam_dakika'].median())

Merkez İlçeler Saatlik Medyan Müdahale Süresi:
saat
0     4.0
1     4.0
2     4.0
3     4.5
4     4.0
5     4.5
6     4.0
7     4.0
8     5.0
9     5.0
10    4.0
11    4.0
12    4.5
13    4.5
14    5.0
15    5.0
16    3.0
17    4.0
18    5.0
19    4.0
20    4.5
21    4.0
22    5.0
23    4.0
Name: toplam_dakika, dtype: float64


In [64]:
print(df_trafik.groupby('saat')['toplam_dakika'].std())

saat
0      4.387245
1      3.712672
2      2.735666
3      3.440930
4      3.287823
5      7.645686
6      3.205441
7      8.679045
8      7.207875
9      2.909044
10     4.631591
11     5.013364
12     3.437464
13     2.791434
14    11.386206
15     4.122534
16     5.551321
17     4.918561
18     6.706940
19     3.855363
20     3.299473
21     3.389274
22     4.576483
23     3.800994
Name: toplam_dakika, dtype: float64


In [65]:
# Sadece saat 14'teki trafik kazalarını görelim
df_14 = df_trafik[df_trafik['saat'] == 14]

# Bu saatteki sürelerin dağılımına bakalım
print("Saat 14:00 için istatistikler:")
print(df_14['toplam_dakika'].describe())

# En uzun süren 5 vakaya bakalım (Outlier tespiti)
print("\nEn uzun süren 14:00 vakaları:")
print(df_14.sort_values(by='toplam_dakika', ascending=False).head())

Saat 14:00 için istatistikler:
count    47.000000
mean      7.468085
std      11.386206
min       1.000000
25%       4.000000
50%       5.000000
75%       7.000000
max      80.000000
Name: toplam_dakika, dtype: float64

En uzun süren 14:00 vakaları:
                    tarih  kurtarma_tipi cikis_saati varis_suresi  \
692   2024-08-04 00:00:00  TRAFİK KAZASI       14:40        01:20   
616   2024-07-15 00:00:00  TRAFİK KAZASI       14:04        00:20   
1088  2024-12-18 00:00:00  TRAFİK KAZASI       14:01        00:14   
407   2024-05-16 00:00:00  TRAFİK KAZASI       14:33        00:12   
859   2024-09-25 00:00:00  TRAFİK KAZASI       14:34        00:12   

           mahalle    ilce  toplam_dakika  saat  
692        ARTICAK  ÖDEMİŞ             80    14  
616      KURUÇEŞME    BUCA             20    14  
1088        CUMALI   KINIK             14    14  
407         BELEVİ  SELÇUK             12    14  
859   KIZILCAHAVLU    TİRE             12    14  
